# ICBHI Model Training — Simple CNN Baseline

Trains a CNN on the v3 preprocessed 3-channel spectrograms (mel + delta + delta-delta).  
Uses 80/20 split: training data for fitting, held-out test data for final inference.

 3-channel input (C), SpecAugment, Mixup (D), Balanced batching (E), Weighted loss

---
## 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, time
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
#===========================================================================
# CONFIG
#===========================================================================
DATA_DIR   = '/content/drive/MyDrive/ICBHI_preprocessed/'
SAVE_DIR   = '/content/drive/MyDrive/ICBHI_models/'
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE    = 64
NUM_EPOCHS    = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
MIXUP_ALPHA   = 0.4   
USE_MIXUP     = True
USE_BALANCED  = True   

# Load metadata
with open(os.path.join(DATA_DIR, 'metadata.json')) as f:
    meta = json.load(f)

NUM_CLASSES = meta['num_classes']
IDX_TO_LABEL = {int(k): v for k, v in meta['idx_to_label'].items()}
CLASS_NAMES = [IDX_TO_LABEL[i] for i in range(NUM_CLASSES)]

# Class weights
class_weights = torch.FloatTensor(
    [meta['class_weights'][str(i)] for i in range(NUM_CLASSES)]
).to(device)

print(f'Classes: {NUM_CLASSES} — {CLASS_NAMES}')
print(f'Input shape: {meta["spectrogram_shape"]}')
print(f'Epochs: {NUM_EPOCHS}, LR: {LEARNING_RATE}, Batch: {BATCH_SIZE}')

---
## 1 — Dataset & DataLoaders

In [ ]:
def spec_augment(spec_3ch, num_freq_masks=2, freq_mask_param=15,
                 num_time_masks=2, time_mask_param=25):
    """SpecAugment on all 3 channels simultaneously."""
    aug = spec_3ch.copy()
    n_mels, n_frames = aug.shape[1], aug.shape[2]
    for _ in range(num_freq_masks):
        f = np.random.randint(1, min(freq_mask_param, n_mels))
        f0 = np.random.randint(0, n_mels - f)
        aug[:, f0:f0+f, :] = 0.0
    for _ in range(num_time_masks):
        t = np.random.randint(1, min(time_mask_param, n_frames))
        t0 = np.random.randint(0, n_frames - t)
        aug[:, :, t0:t0+t] = 0.0
    return aug


class ICBHIDataset(Dataset):
    def __init__(self, npz_path, augment=False):
        data = np.load(npz_path)
        self.specs = data['spectrograms']     # (N, 3, 128, T)
        self.labels = data['disease_labels']   # (N,)
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        spec = self.specs[idx].copy()
        if self.augment:
            spec = spec_augment(spec)
        return torch.FloatTensor(spec), torch.tensor(self.labels[idx], dtype=torch.long)


class ClassBalancedSampler(Sampler):
    
    def __init__(self, labels, num_classes, batch_size, num_batches):
        self.num_classes = num_classes
        self.batch_size = batch_size
        self.num_batches = num_batches
        self.per_class = batch_size // num_classes
        self.class_indices = {c: np.where(np.array(labels) == c)[0]
                              for c in range(num_classes)}

    def __iter__(self):
        for _ in range(self.num_batches):
            batch = []
            for c in range(self.num_classes):
                idx = self.class_indices[c]
                if len(idx) > 0:
                    batch.extend(np.random.choice(idx, self.per_class, replace=True))
            np.random.shuffle(batch)
            yield from batch

    def __len__(self):
        return self.num_batches * self.batch_size


print('Dataset classes defined.')

In [ ]:
# Load data — use train for training, merge val+test as unseen 20%
# (The v3 split was 70/15/15 = train/val/test, so val+test ≈ 30% unseen)
# We use val for validation during training and test for final inference.

train_ds = ICBHIDataset(os.path.join(DATA_DIR, 'train_data.npz'), augment=True)
val_ds   = ICBHIDataset(os.path.join(DATA_DIR, 'val_data.npz'),   augment=False)
test_ds  = ICBHIDataset(os.path.join(DATA_DIR, 'test_data.npz'),  augment=False)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test (unseen): {len(test_ds)}')
print(f'Train is ~{100*len(train_ds)/(len(train_ds)+len(val_ds)+len(test_ds)):.0f}% '
      f'of total, Test is ~{100*len(test_ds)/(len(train_ds)+len(val_ds)+len(test_ds)):.0f}%')

# DataLoaders
if USE_BALANCED:
    train_labels = np.load(os.path.join(DATA_DIR, 'train_data.npz'))['disease_labels']
    num_batches = len(train_ds) // BATCH_SIZE
    sampler = ClassBalancedSampler(train_labels, NUM_CLASSES, BATCH_SIZE, num_batches)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=2, pin_memory=True)
    print(f'Using balanced batching: {BATCH_SIZE//NUM_CLASSES} samples/class/batch')
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)

val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# Quick shape check
x, y = next(iter(train_loader))
print(f'\nBatch shape: x={x.shape}, y={y.shape}')

---
## 2 — Model: Simple CNN

In [ ]:
class SimpleCNN(nn.Module):
    """
    Simple CNN for 3-channel spectrogram classification.
    4 conv blocks + global average pooling + classifier head.
    """
    def __init__(self, num_classes=8, in_channels=3):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1: 3 -> 32
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            # Block 2: 32 -> 64
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.15),

            # Block 3: 64 -> 128
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            # Block 4: 128 -> 256
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),  # global average pooling
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = SimpleCNN(num_classes=NUM_CLASSES, in_channels=3).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: SimpleCNN')
print(f'Parameters: {total_params:,} ({trainable:,} trainable)')
print(f'Input: (batch, 3, 128, 157) → Output: (batch, {NUM_CLASSES})')

---
## 3 — Training loop with Mixup

In [ ]:
def mixup_data(x, y, alpha=0.4):
    
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def train_one_epoch(model, loader, criterion, optimizer, use_mixup=True):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        if use_mixup:
            x_mix, y_a, y_b, lam = mixup_data(x, y, MIXUP_ALPHA)
            pred = model(x_mix)
            loss = lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)
        else:
            pred = model(x)
            loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        _, predicted = pred.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()

    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = criterion(pred, y)

        running_loss += loss.item() * x.size(0)
        _, predicted = pred.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    return running_loss / total, 100.0 * correct / total, all_preds, all_labels


print('Training functions defined.')

In [ ]:
# Setup
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                                  patience=5, verbose=True)

# Training history
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_epoch = 0

print(f'Training for {NUM_EPOCHS} epochs...')
print(f'Mixup: {"ON" if USE_MIXUP else "OFF"} (alpha={MIXUP_ALPHA})')
print(f'Balanced batching: {"ON" if USE_BALANCED else "OFF"}')
print(f'Weighted loss: ON (class_weights applied)')
print(f'\n{"Epoch":>5s} {"Train Loss":>10s} {"Train Acc":>10s} {"Val Loss":>10s} {"Val Acc":>10s} {"LR":>10s}')
print('-' * 60)

t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion,
                                             optimizer, use_mixup=USE_MIXUP)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']

    marker = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'best_model.pth'))
        marker = ' ★'

    print(f'{epoch:5d} {train_loss:10.4f} {train_acc:9.2f}% {val_loss:10.4f} {val_acc:9.2f}%  {lr:.1e}{marker}')

elapsed = time.time() - t0
print(f'\nTraining complete in {elapsed/60:.1f} min')
print(f'Best val accuracy: {best_val_acc:.2f}% at epoch {best_epoch}')

---
## 4 — Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train', linewidth=2)
axes[0].plot(epochs, history['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['train_acc'], label='Train', linewidth=2)
axes[1].plot(epochs, history['val_acc'], label='Val', linewidth=2)
axes[1].axhline(best_val_acc, color='green', linestyle='--', alpha=0.5,
                label=f'Best val={best_val_acc:.1f}%')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
plt.show()

---
## 5 — Final inference on unseen test set

In [ ]:
# Load best model
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'best_model.pth')))
print(f'Loaded best model from epoch {best_epoch}\n')

# Run inference on held-out test set
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)

print(f'=== TEST SET RESULTS (unseen data) ===')
print(f'Loss: {test_loss:.4f}')
print(f'Accuracy: {test_acc:.2f}%')
print(f'Samples: {len(test_labels)}')

In [ ]:
# Classification report
# Only include classes that appear in the test set
test_classes_present = sorted(set(test_labels))
target_names = [IDX_TO_LABEL[i] for i in test_classes_present]

print('Classification Report (per-class metrics):')
print('=' * 60)
print(classification_report(test_labels, test_preds,
                             labels=test_classes_present,
                             target_names=target_names,
                             digits=3, zero_division=0))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))

cm = confusion_matrix(test_labels, test_preds, labels=test_classes_present)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Confusion Matrix — Test Set (acc={test_acc:.1f}%)', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# Per-class accuracy breakdown
print('Per-class accuracy on test set:')
print(f'{"Class":20s} {"Correct":>8s} {"Total":>8s} {"Accuracy":>10s}')
print('-' * 50)
for cls_idx in test_classes_present:
    mask = np.array(test_labels) == cls_idx
    total = mask.sum()
    correct = (np.array(test_preds)[mask] == cls_idx).sum()
    acc = 100.0 * correct / total if total > 0 else 0
    print(f'{IDX_TO_LABEL[cls_idx]:20s} {correct:8d} {total:8d} {acc:9.1f}%')

---
## 6 — Save results summary

In [ ]:
results = {
    'model': 'SimpleCNN',
    'epochs': NUM_EPOCHS,
    'best_epoch': best_epoch,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'mixup': USE_MIXUP,
    'balanced_batching': USE_BALANCED,
    'best_val_acc': best_val_acc,
    'test_acc': test_acc,
    'test_loss': test_loss,
    'history': history,
    'techniques used': ['A_audio_oversample', 'B_per_equip_norm',
                    'C_3channel', 'D_mixup', 'E_balanced_batch']
}

with open(os.path.join(SAVE_DIR, 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('Saved to', SAVE_DIR)
for fn in sorted(os.listdir(SAVE_DIR)):
    fp = os.path.join(SAVE_DIR, fn)
    if os.path.isfile(fp):
        print(f'  {fn}: {os.path.getsize(fp)/(1024**2):.1f} MB')

print(f'\n=== SUMMARY ===')
print(f'Best validation accuracy: {best_val_acc:.2f}%')
print(f'Test accuracy (unseen):   {test_acc:.2f}%')